### University of Virginia
### DS 5110: Big Data Systems

### Lab: Supervised Learning
### Last updated: March 2, 2023

---

#### Instructions

This project has two parts:
- Part I: Classification - build and apply a logistic regression model on the Wisconsin Breast Cancer dataset.
- Part II: Regression - build and apply a linear regression model on the California Housing dataset.

**Total Possible Points: 10**

---

#### Part I: Classification (5 POINTS)

Here are the specifications and grading breakdown:

- the target variable is `diagnosis`
- use `f1`, `f2` as predictors (1 PT)
- split data into 60% training set, 40% test set 
- standardize the predictors (1 PT)
- use seed=314 whenever a seed is needed
- fit a Logistic Regression model with an intercept (1 PT)
- compute and show the area under the ROC curve for the test set (2 PTS)

In [51]:
# Laad a Spark session.
from pyspark.sql import SparkSession

DATA_FILEPATH = 'wisc_breast_cancer_w_fields.csv'

spark = SparkSession \
    .builder \
    .appName("Wisc BRCA") \
    .getOrCreate()

In [52]:
# Supproess the annoying warn message.
spark.sparkContext.setLogLevel("ERROR")

# Load the data into a Spark DataFrame.
df_wisc = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load("/standard/ds7200-apt4c/large_datasets/ds5110/06_foundations_of_mllib/wisc_breast_cancer_w_fields.csv")

# Reset the message level to allow warnings.
spark.sparkContext.setLogLevel("WARN")

In [53]:
# Inspect the dataset.
#df_wisc.printSchema()
df_wisc.show(5)
df_wisc.count()

+--------+---------+-----+-----+-----+------+-------+-------+------+-------+------+-------+------+------+-----+-----+--------+-------+-------+-------+-------+--------+-----+-----+-----+------+------+------+------+------+------+-------+
|      id|diagnosis|   f1|   f2|   f3|    f4|     f5|     f6|    f7|     f8|    f9|    f10|   f11|   f12|  f13|  f14|     f15|    f16|    f17|    f18|    f19|     f20|  f21|  f22|  f23|   f24|   f25|   f26|   f27|   f28|   f29|    f30|
+--------+---------+-----+-----+-----+------+-------+-------+------+-------+------+-------+------+------+-----+-----+--------+-------+-------+-------+-------+--------+-----+-----+-----+------+------+------+------+------+------+-------+
|  842302|        M|17.99|10.38|122.8|1001.0| 0.1184| 0.2776|0.3001| 0.1471|0.2419|0.07871| 1.095|0.9053|8.589|153.4|0.006399|0.04904|0.05373|0.01587|0.03003|0.006193|25.38|17.33|184.6|2019.0|0.1622|0.6656|0.7119|0.2654|0.4601| 0.1189|
|  842517|        M|20.57|17.77|132.9|1326.0|0.08474|0.0

569

#### Enter code and solution

In [54]:
# Register the DF as a temp view first.
df_wisc.createOrReplaceTempView("df_wisc")

# Select the columns needed for this analysis.
df_wisc_data = spark.sql("""
SELECT f1, f2, diagnosis
FROM df_wisc
""")

# Select the first five records.
# Note that diagnosis is string column; this will be converted to numeric shortly.
df_wisc_data.show(5)

+-----+-----+---------+
|   f1|   f2|diagnosis|
+-----+-----+---------+
|17.99|10.38|        M|
|20.57|17.77|        M|
|19.69|21.25|        M|
|11.42|20.38|        M|
|20.29|14.34|        M|
+-----+-----+---------+
only showing top 5 rows


In [55]:
# Split the dataset.
trainDF, testDF = df_wisc_data.randomSplit([.6, .4], seed=314)
print(f"""There are {trainDF.count()} rows in the training set, and {testDF.count()} in the test set.""")

There are 354 rows in the training set, and 215 in the test set.


In [56]:
# Use StringIndexer to convert diagnosis into a numeric column.
from pyspark.ml.feature import StringIndexer
indexer = StringIndexer(inputCol="diagnosis", outputCol="diagnosis_ind")

# Update the train data set to have this indexed column.
indexer_model = indexer.fit(trainDF)
trainDF = indexer_model.transform(trainDF)

# Also, apply the diagnosis label numeric encoding mapping learned from the trainDF to testDF.
testDF = indexer_model.transform(testDF)

In [57]:
# Look at a few records to see what this did.
trainDF.show(5)
testDF.show(5)

+-----+-----+---------+-------------+
|   f1|   f2|diagnosis|diagnosis_ind|
+-----+-----+---------+-------------+
|6.981|13.43|        B|          0.0|
|7.691|25.44|        B|          0.0|
|7.729|25.49|        B|          0.0|
|8.196|16.84|        B|          0.0|
|8.571| 13.1|        B|          0.0|
+-----+-----+---------+-------------+
only showing top 5 rows
+-----+-----+---------+-------------+
|   f1|   f2|diagnosis|diagnosis_ind|
+-----+-----+---------+-------------+
| 7.76|24.54|        B|          0.0|
|8.219| 20.7|        B|          0.0|
|8.598|20.98|        B|          0.0|
|8.618|11.79|        B|          0.0|
|8.726|15.83|        B|          0.0|
+-----+-----+---------+-------------+
only showing top 5 rows


In [58]:
# Use VectorAssembler to package f1, f2 into a feature column.
from pyspark.ml.feature import VectorAssembler

vecAssembler = VectorAssembler(inputCols=["f1","f2"], outputCol="features")
vecTrainDF = vecAssembler.transform(trainDF)
vecTrainDF.select("*").show(10)

+-----+-----+---------+-------------+-------------+
|   f1|   f2|diagnosis|diagnosis_ind|     features|
+-----+-----+---------+-------------+-------------+
|6.981|13.43|        B|          0.0|[6.981,13.43]|
|7.691|25.44|        B|          0.0|[7.691,25.44]|
|7.729|25.49|        B|          0.0|[7.729,25.49]|
|8.196|16.84|        B|          0.0|[8.196,16.84]|
|8.571| 13.1|        B|          0.0| [8.571,13.1]|
|8.597| 18.6|        B|          0.0| [8.597,18.6]|
|8.671|14.45|        B|          0.0|[8.671,14.45]|
|8.734|16.84|        B|          0.0|[8.734,16.84]|
|8.878|15.49|        B|          0.0|[8.878,15.49]|
|8.888|14.64|        B|          0.0|[8.888,14.64]|
+-----+-----+---------+-------------+-------------+
only showing top 10 rows


In [66]:
# Apply the assembler to testDF as wel;/
vecTestDF = vecAssembler.transform(testDF)
vecTestDF.select("*").show(10)

+-----+-----+---------+-------------+-------------+
|   f1|   f2|diagnosis|diagnosis_ind|     features|
+-----+-----+---------+-------------+-------------+
| 7.76|24.54|        B|          0.0| [7.76,24.54]|
|8.219| 20.7|        B|          0.0| [8.219,20.7]|
|8.598|20.98|        B|          0.0|[8.598,20.98]|
|8.618|11.79|        B|          0.0|[8.618,11.79]|
|8.726|15.83|        B|          0.0|[8.726,15.83]|
|9.333|21.94|        B|          0.0|[9.333,21.94]|
|9.423|27.88|        B|          0.0|[9.423,27.88]|
|9.567|15.91|        B|          0.0|[9.567,15.91]|
|9.742|15.67|        B|          0.0|[9.742,15.67]|
|9.755| 28.2|        B|          0.0| [9.755,28.2]|
+-----+-----+---------+-------------+-------------+
only showing top 10 rows


In [67]:
# Standardize the predictors for both the train and test data sets.
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")
scalerModel = scaler.fit(vecTrainDF)
scaledTrain = scalerModel.transform(vecTrainDF)
scaledTest = scalerModel.transform(vecTestDF)

In [60]:
# Fit a logistic regression model.
from pyspark.ml.classification import LogisticRegression

# Instantiate the model.
lr = LogisticRegression(
    labelCol="diagnosis_ind",
    featuresCol='scaledFeatures',
    maxIter=10,
    regParam=0.3,
    elasticNetParam=0.8)

# Fit the model.
lrModel = lr.fit(scaledTrain)

# Print the coefficients and intercept.
print("The coefficients are: " + str(lrModel.coefficients))
print("The intercept is: " + str(lrModel.intercept))

The coefficients are: [0.37279162798258053,0.0]
The intercept is: -2.0190019439808675


In [68]:
# Compute and show the area under the ROC curve for the test set.
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Compute the predictions.
lrPredTest = lrModel.transform(scaledTest)
lrPredTest.select('probability', 'prediction').show(5, truncate=False)

+----------------------------------------+----------+
|probability                             |prediction|
+----------------------------------------+----------+
|[0.7703641210045565,0.22963587899544347]|0.0       |
|[0.7617935964675805,0.2382064035324195] |0.0       |
|[0.7545529620533489,0.24544703794665113]|0.0       |
|[0.7541667748560446,0.24583322514395545]|0.0       |
|[0.7520742952478074,0.24792570475219255]|0.0       |
+----------------------------------------+----------+
only showing top 5 rows


In [69]:
# Setup the evaluator.
evaluator = BinaryClassificationEvaluator(
    rawPredictionCol="rawPrediction",
    labelCol="diagnosis_ind",
    metricName="areaUnderPR")

# Pass the predictions to the evaluator.
roc_auc = evaluator.evaluate(lrPredTest)
print("Area under PR Curve:", roc_auc)

Area under PR Curve: 0.9317158305650639


#### Part II: Regression (5 POINTS)

In this project, you will work with the California Home Price dataset to train a regression model and predict median home prices. Here are the specifications and grading breakdown:

- Scale the response variable median_house_value, dividing by 100000 (1 PT)

- Split data into train set (80%), test set (20%) using seed=314 (1 PT)

- Add new predictor: `rooms_per_household`

- In the training set, select all of these features and standardize them: (1 PT)

feats = ["total_bedrooms", 
         "population", 
         "households", 
         "median_income", 
         "rooms_per_household"]

- Fit a linear regression model on the training set with these parameters:

  - maxIter=10
  - regParam=0.3
  - elasticNetParam=0.8  


- Compute the MSE on the test set (2 PTS)

In [1]:
import os
import pandas as pd

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

/opt/conda/lib/python3.7/site-packages/pyspark/bin/load-spark-env.sh: line 68: ps: command not found
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


23/03/03 02:27:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
23/03/03 02:27:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
DATA_FILEPATH2 = 'cal_housing_data_preproc_w_header.txt'

#### Enter code and solution